# ML-04 â€” Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** â€” each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**Data Contract (Plain Words):**
- **Grain:** One row represents the performance of a specific pseudonymized content item on a specific day for a specific client (`report_date` × `client_hash_id` × `content_hash_id`).
- **Tables:** We will use `fact_content_daily_performance` for time-series metrics.
- **Time Window:** We will use a mid-panel month: **March 2026** (`month=2026-03`) to avoid testing on the final outcome month.
- **Label / Proxy:** We will predict whether a page will experience a significant drop in impressions in the next 30 days (Content Decline/Refresh Opportunity), derived from trend indicators.
- **Excluded:** We deliberately exclude data from clients before their `ga4_data_start` date because GA4 metrics are zero-filled, which would be falsely interpreted as "no engagement".

In [1]:
import duckdb
import pandas as pd

# Connect and authenticate
con = duckdb.connect()
con.execute("CREATE SECRET (TYPE huggingface, TOKEN '<YOUR_HUGGINGFACE_TOKEN>')")
rel = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"

# Query 1: Prove the Grain (No duplicates per report_date, client_hash_id, content_hash_id)
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) as count 
    FROM read_parquet('{rel}') 
    GROUP BY 1, 2, 3 
    HAVING COUNT(*) > 1 
    LIMIT 5
""").df()

if grain_check.empty:
    print("Grain Check: PASS (0 duplicate rows found at this grain)")
else:
    print("Grain Check: FAIL (Duplicates found)")


Grain Check: PASS (0 duplicate rows found at this grain)


## 2. Fields: feature / label / context / excluded

**5 Features (Available at Decision Moment):**
1. `impressions_90d` (total_impressions_mar26): Knowable at the decision moment because it aggregates historical Search Console data from the prior month.
2. `clicks_90d` (total_clicks_mar26): Knowable at the decision moment because past clicks are logged and finalized.
3. `avg_position` (average_position_mar26): Knowable at the decision moment because ranking position is captured daily.
4. `sessions_90d` (engagement_rate_mar26): Knowable at the decision moment because it relies only on recorded GA4 sessions from the past month.
5. `engagement_time` (ga4_total_engagement_sec): Knowable at the decision moment because historical engagement seconds are already logged and finalized.

*Below, we build a small feature frame showing these columns.*

In [2]:
# Building a small feature frame for our lane (Top 5 rows)
feature_frame = con.sql(f"""
    SELECT 
        content_hash_id,
        SUM(gsc_impressions) as impressions_90d,
        SUM(gsc_clicks) as clicks_90d,
        AVG(gsc_avg_position) as avg_position,
        SUM(ga4_sessions) as sessions_90d,
        SUM(ga4_total_engagement_sec) as engagement_time
    FROM read_parquet('{rel}')
    WHERE ga4_data_available IS TRUE
    GROUP BY content_hash_id
    LIMIT 5
""").df()

print("Feature Frame Preview (5 rows):")
print(feature_frame)


Feature Frame Preview (5 rows):
            content_hash_id  impressions_90d  clicks_90d  avg_position  \
0  content_374cf959e66baff7            183.0         1.0     35.469077   
1  content_448878c5eb8adc78           6680.0         6.0     29.044922   
2  content_853134d4eb6f66fe           4992.0        34.0     12.098136   
3  content_2826a33c387e5f5b             44.0         0.0     10.477273   
4  content_9dcaefd5e410fa14          14532.0        33.0      4.999256   

   sessions_90d  engagement_time  
0           5.0            638.0  
1          30.0             97.0  
2          43.0            318.0  
3           1.0              0.0  
4          31.0            206.0  


## 3. Verify it with queries (grain, counts, missing values, windows)

*Queries are split across the code blocks. Below we check Availability (IS TRUE) and demonstrate the Leakage Trap.*

In [3]:
# Query 3: Availability (Filtering with IS TRUE)
# Let's see how many rows survive if we only look at data where GA4 is fully available
availability_check = con.sql(f"""
    SELECT COUNT(*) as valid_rows 
    FROM read_parquet('{rel}')
    WHERE ga4_data_available IS TRUE
""").df()
print(f"Rows with GA4 Data Available: {availability_check['valid_rows'].iloc[0]:,}")

# --- THE LEAKAGE TRAP ---
print("\n--- THE LEAKAGE TRAP ---")
print("If we accidentally use a label-derived column (like predicting if CTR is low using 'clicks / impressions' from the SAME window), our model is cheating.")
# We will simulate the trap locally on a tiny dataframe
trap_df = pd.DataFrame({'clicks': [10, 20, 0], 'impressions': [100, 200, 50]})
# The Trap Feature:
trap_df['future_ctr'] = trap_df['clicks'] / trap_df['impressions']
print("Added trap feature 'future_ctr'. Model accuracy would jump to 100% because the feature IS the label.")
# Delete it to keep honest numbers
del trap_df['future_ctr']
print("Trap removed. Honest modeling restored.")


Rows with GA4 Data Available: 413,966

--- THE LEAKAGE TRAP ---
If we accidentally use a label-derived column (like predicting if CTR is low using 'clicks / impressions' from the SAME window), our model is cheating.
Added trap feature 'future_ctr'. Model accuracy would jump to 100% because the feature IS the label.
Trap removed. Honest modeling restored.


## 4. Data limits

**Limitation of this Slice:**
This slice (March 2026) cannot tell us anything about the long-term historical seasonality (e.g., Q4 spikes) because it is limited to a single month. Additionally, as noted in the data dictionary, rows before a client's `ga4_data_start` have zero-filled GA4 columns, meaning we must actively filter by `ga4_data_available` to avoid treating missing data as zero engagement.

In [4]:
print("All contract checks complete. Ready for modeling.")

All contract checks complete. Ready for modeling.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled â€” markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime â†’ Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` â€” then submit your repo URL on the card. Done.